In [1]:
import Helper as req
import json
import pandas as pd
from pandas import DataFrame
import requests


In [26]:

# GLOBAL VARIABLES

matches_file = "matches.json"
data_file = "data.pkl"
results_file = "results.json"
api_key = "RGAPI-987c33b6-6a1e-4a6f-93a4-a8edef2bd4ae"
riot_id = "Ferix8475#NA1"

gameName = riot_id.split("#")[0]
tagline = riot_id.split("#")[1]

puuid = req.fetch_account_puuid(gameName, tagline, api_key)


# UPDATE THE DATAFRAME
req.update_data(puuid=puuid, api_key=api_key)

df = pd.read_pickle(data_file)

In [40]:
def purge_df(df : DataFrame):
    df = df[~df.isin(['']).any(axis=1)]
    df = df.dropna()
    return df

In [41]:
objective_df = df.groupby(['Champion', 'Role', 'Win']).agg({
    'Barons_Killed': 'mean',
    'Void_Grubs_Killed': 'mean',
    'Dragons_Killed': 'mean',
    'Turret_Killed': 'mean',
    'Rift_Heralds_Killed': 'mean',

}).reset_index()
objective_df = purge_df(objective_df)
objective_df

,Champion,Role,Win,Barons_Killed,Void_Grubs_Killed,Dragons_Killed,Turret_Killed,Rift_Heralds_Killed
2,Aatrox,JUNGLE,False,0.000000,1.2,0.800000,0.400000,0.400000
3,Aatrox,JUNGLE,True,0.500000,2.5,2.500000,1.000000,0.000000
4,Aatrox,MIDDLE,False,0.000000,0.0,0.000000,0.500000,0.000000
5,Aatrox,MIDDLE,True,0.500000,1.0,3.500000,4.000000,0.500000
6,Aatrox,TOP,False,0.214286,0.928571,1.285714,1.285714,0.500000
...,...,...,...,...,...,...,...,...
167,Zed,MIDDLE,True,0.500000,1.903226,1.916667,2.500000,0.555556
168,Zed,TOP,False,0.000000,0.0,1.000000,1.000000,0.000000
169,Zed,TOP,True,0.333333,5.0,1.666667,3.333333,0.666667
170,Zed,UTILITY,False,0.000000,0.0,1.000000,0.000000,1.000000


In [43]:
grouped = df.groupby(['Champion', 'Role', 'Win']).size().unstack(fill_value=0)
grouped['Winrate'] = grouped[True] / (grouped[True] + grouped[False]) * 100
grouped['Games Played'] = (grouped[True] + grouped[False]) 
general_df = grouped[['Winrate', 'Games Played']].reset_index()
general_df = purge_df(general_df)
general_df

Win,Champion,Role,Winrate,Games Played
1,Aatrox,JUNGLE,28.571429,7
2,Aatrox,MIDDLE,50.000000,4
3,Aatrox,TOP,44.000000,25
4,Aatrox,UTILITY,30.769231,13
5,Ahri,MIDDLE,50.000000,2
...,...,...,...,...
112,Zac,UTILITY,33.333333,3
114,Zed,JUNGLE,55.172414,58
115,Zed,MIDDLE,54.545455,66
116,Zed,TOP,75.000000,4


In [29]:
effectiveness_df = df.groupby(['Champion', 'Role', 'Win']).agg({
        'Total_Minions_Killed': 'mean',
        'Total_Jungle_Monsters_Killed': 'mean',
        'Total_Damage_DealtToChampions': 'mean',
        'KDA': 'mean',
        'Kill_Participation': 'mean',
        'Damage_Share': 'mean',
        'Turret_Plates_Taken': 'mean',
        'Gold_Per_Minute': 'mean',
        'Damage_Per_Minute': 'mean',
        'Vision_Score_Per_Minute': 'mean',
        'Lane_Minions_Before_10_Minutes': 'mean',
        'Jungle_CS_Before_10_Minutes': 'mean',
        'Sol_Kills':'mean'

}).reset_index()
effectiveness_df = purge_df(effectiveness_df)

effectiveness_df


,Champion,Role,Win,Total_Minions_Killed,Total_Jungle_Monsters_Killed,Total_Damage_DealtToChampions,KDA,Kill_Participation,Damage_Share,Turret_Plates_Taken,Gold_Per_Minute,Damage_Per_Minute,Vision_Score_Per_Minute,Lane_Minions_Before_10_Minutes,Jungle_CS_Before_10_Minutes,Sol_Kills
2,Aatrox,JUNGLE,False,17.400000,107.800000,6251.800000,1.233333,0.286486,0.106599,0.4,336.544357,238.364878,0.723166,0.0,54.110000,0.0
3,Aatrox,JUNGLE,True,32.000000,143.000000,17619.500000,3.166667,0.357558,0.162618,0.0,391.677828,474.186637,0.823045,2.0,56.500000,1.5
4,Aatrox,MIDDLE,False,116.000000,0.000000,12644.000000,0.902778,0.291379,0.214181,0.5,385.095136,561.395316,0.643855,51.5,0.000000,1.0
5,Aatrox,MIDDLE,True,203.500000,15.000000,40967.500000,3.811111,0.479167,0.237691,1.5,469.629052,1051.767274,0.868906,59.5,0.000000,1.5
6,Aatrox,TOP,False,177.142857,7.000000,21756.214286,1.387915,0.337247,0.225665,1.357143,358.042213,650.800007,0.644207,56.285714,0.000000,2.142857
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
167,Zed,MIDDLE,True,193.583333,6.694444,27092.222222,4.376786,0.418480,0.247488,2.611111,473.071785,843.890990,0.748804,66.25,0.088889,3.638889
168,Zed,TOP,False,211.000000,5.000000,21874.000000,0.400000,0.181818,0.183768,1.0,320.356173,584.754435,0.835389,59.0,0.000000,2.0
169,Zed,TOP,True,239.000000,38.333333,65826.333333,5.933333,0.555820,0.341064,3.333333,569.965314,1631.315158,0.928978,79.0,0.000000,6.0
170,Zed,UTILITY,False,53.000000,0.000000,20131.000000,1.428571,0.454545,0.248205,1.0,331.607087,757.125193,2.218990,14.0,0.000000,2.0


In [31]:
winrate_by_role = df.groupby(['Champion', 'Role']).agg(
    Winrate=('Win', 'mean'),
    Games_Played=('Win', 'count')
).reset_index()
winrate_by_role = purge_df(winrate_by_role)
winrate_by_role

,Champion,Role,Winrate,Games_Played
1,Aatrox,JUNGLE,0.285714,7
2,Aatrox,MIDDLE,0.500000,4
3,Aatrox,TOP,0.440000,25
4,Aatrox,UTILITY,0.307692,13
5,Ahri,MIDDLE,0.500000,2
...,...,...,...,...
112,Zac,UTILITY,0.333333,3
114,Zed,JUNGLE,0.551724,58
115,Zed,MIDDLE,0.545455,66
116,Zed,TOP,0.750000,4


Win,Champion,Role,Winrate,Games Played
1,Aatrox,JUNGLE,28.571429,7
2,Aatrox,MIDDLE,50.000000,4
3,Aatrox,TOP,44.000000,25
4,Aatrox,UTILITY,30.769231,13
5,Ahri,MIDDLE,50.000000,2
...,...,...,...,...
112,Zac,UTILITY,33.333333,3
114,Zed,JUNGLE,55.172414,58
115,Zed,MIDDLE,54.545455,66
116,Zed,TOP,75.000000,4


In [50]:
# Calculate Winrates by Item and Champion
melted_df = df.melt(id_vars=['Champion', 'Win'], value_vars=['Item0', 'Item1', 'Item2', 'Item3', 'Item4', 'Item5', 'Item6'],
                    var_name='ItemSlot', value_name='Item')

melted_df = melted_df[melted_df['Item'] != 0] # Get rid of empty items

item_winrate_df = melted_df.groupby(['Champion', 'Item']).agg(
    Winrate=('Win', 'mean'),
    Games_Played=('Win', 'count')
).reset_index()

mydict = req.json_extract_important_items()

items_to_keep = list(mydict.keys())
item_winrate_df = item_winrate_df[item_winrate_df['Item'].isin(items_to_keep)] # Get rid of unwanted items (components, epic items)

item_winrate_df['Item'] = item_winrate_df['Item'].replace(mydict) # Replace IDs with Names
item_winrate_df = purge_df(item_winrate_df)
item_winrate_df

,Champion,Item,Winrate,Games_Played
6,Aatrox,Doran's Shield,0.222222,9
7,Aatrox,Doran's Blade,0.500000,4
13,Aatrox,Control Ward,0.200000,5
14,Aatrox,Evenshroud,0.000000,1
15,Aatrox,Guardian Angel,1.000000,2
...,...,...,...,...
932,Zed,Axiom Arc,0.785714,14
933,Zed,Hubris,0.818182,11
934,Zed,Profane Hydra,0.634615,52
935,Zed,Voltaic Cyclosword,0.660000,50


In [15]:
runes = req.json_extract_runes()
treepage = {
    8000: "Precision",
    8100: "Domination",
    8200: "Sorcery",
    8300: "Inspiration",
    8400: "Resolve"
}


df['Primary_Tree'] = df['Primary_Tree'].replace(treepage)
df['Secondary_Tree'] = df['Secondary_Tree'].replace(treepage)
df['Primary_Keystone'] = df['Primary_Keystone'].replace(runes)
runes

{8369: 'First Strike',
 8446: 'Demolish',
 8126: 'Cheap Shot',
 8415: 'The Arcane Colossus',
 8410: 'Approach Velocity',
 8232: 'Waterwalking',
 8299: 'Last Stand',
 8112: 'Electrocute',
 8234: 'Celerity',
 8453: 'Revitalize',
 8360: 'Unsealed Spellbook',
 8004: 'The Brazen Perfect',
 8128: 'Dark Harvest',
 8220: 'The Calamity',
 8016: 'The Merciless Elite',
 8473: 'Bone Plating',
 8339: 'Celestial Body',
 8214: 'Summon Aery',
 8237: 'Scorch',
 8139: 'Taste of Blood',
 9101: 'Absorb Life',
 8008: 'Lethal Tempo',
 8010: 'Conqueror',
 9105: 'Legend: Haste',
 8106: 'Ultimate Hunter',
 8017: 'Cut Down',
 8224: 'Nullifying Orb',
 8210: 'Transcendence',
 8005: 'Press the Attack',
 8435: 'Mirror Shell',
 8115: 'The Aether Blade',
 8359: 'Kleptomancy',
 8352: 'Time Warp Tonic',
 5003: 'Magic Resist',
 8135: 'Treasure Hunter',
 8120: 'Ghost Poro',
 8134: 'Ingenious Hunter',
 8351: 'Glacial Augment',
 8242: 'Unflinching',
 8401: 'Shield Bash',
 9111: 'Triumph',
 8105: 'Relentless Hunter',
 8454:

In [33]:
moregrouped = df.groupby(['Champion', 'Role', 'Primary_Tree', 'Secondary_Tree']).agg(
    Winrate=('Win', 'mean'),
    Games_Played=('Win', 'count')
).reset_index()
moregrouped = purge_df(moregrouped)
moregrouped

,Champion,Role,Primary_Tree,Secondary_Tree,Winrate,Games_Played
1,Aatrox,JUNGLE,8000,8100,0.285714,7
2,Aatrox,MIDDLE,8000,8400,0.500000,4
3,Aatrox,TOP,8000,8100,0.000000,1
4,Aatrox,TOP,8000,8400,0.458333,24
5,Aatrox,UTILITY,8000,8100,0.000000,1
...,...,...,...,...,...,...
176,Zed,MIDDLE,8300,8400,0.000000,1
177,Zed,TOP,8000,8100,1.000000,1
178,Zed,TOP,8100,8200,1.000000,2
179,Zed,TOP,8100,8300,0.000000,1


In [34]:
zed_info = moregrouped[moregrouped['Champion'] == 'Zed']
zed_info

,Champion,Role,Primary_Tree,Secondary_Tree,Winrate,Games_Played
161,Zed,JUNGLE,8000,8100,0.800000,5
162,Zed,JUNGLE,8000,8200,0.000000,4
163,Zed,JUNGLE,8100,8000,0.333333,3
164,Zed,JUNGLE,8100,8200,0.846154,13
165,Zed,JUNGLE,8300,8000,0.500000,12
166,Zed,JUNGLE,8300,8100,0.470588,17
167,Zed,JUNGLE,8300,8200,0.500000,4
168,Zed,MIDDLE,8000,8100,0.333333,6
169,Zed,MIDDLE,8000,8200,0.500000,2
170,Zed,MIDDLE,8100,8200,0.590909,22


In [35]:
keystone_runes_df = df.groupby(['Champion', 'Role', 'Primary_Keystone', 'Secondary_Tree']).agg(
    Winrate=('Win', 'mean'),
    Games_Played=('Win', 'count')
).reset_index()

keystone_runes_df = purge_df(keystone_runes_df)
keystone_runes_df

,Champion,Role,Primary_Keystone,Secondary_Tree,Winrate,Games_Played
1,Aatrox,JUNGLE,8010,8100,0.285714,7
2,Aatrox,MIDDLE,8010,8400,0.500000,4
3,Aatrox,TOP,8010,8100,0.000000,1
4,Aatrox,TOP,8010,8400,0.458333,24
5,Aatrox,UTILITY,8010,8100,0.000000,1
...,...,...,...,...,...,...
179,Zed,MIDDLE,8369,8400,0.000000,1
180,Zed,TOP,8010,8100,1.000000,1
181,Zed,TOP,8112,8200,1.000000,2
182,Zed,TOP,8112,8300,0.000000,1


In [ ]:
print(req.json_extract_runes())
treepage = {
    8000: "Precision",
    8100: "Domination",
    8200: "Sorcery",
    8300: "Inspiration",
    8400: "Resolve"
}

{8369: 'First Strike', 8446: 'Demolish', 8126: 'Cheap Shot', 8415: 'The Arcane Colossus', 8410: 'Approach Velocity', 8232: 'Waterwalking', 8299: 'Last Stand', 8112: 'Electrocute', 8234: 'Celerity', 8453: 'Revitalize', 8360: 'Unsealed Spellbook', 8004: 'The Brazen Perfect', 8128: 'Dark Harvest', 8220: 'The Calamity', 8016: 'The Merciless Elite', 8473: 'Bone Plating', 8339: 'Celestial Body', 8214: 'Summon Aery', 8237: 'Scorch', 8139: 'Taste of Blood', 9101: 'Absorb Life', 8008: 'Lethal Tempo', 8010: 'Conqueror', 9105: 'Legend: Haste', 8106: 'Ultimate Hunter', 8017: 'Cut Down', 8224: 'Nullifying Orb', 8210: 'Transcendence', 8005: 'Press the Attack', 8435: 'Mirror Shell', 8115: 'The Aether Blade', 8359: 'Kleptomancy', 8352: 'Time Warp Tonic', 5003: 'Magic Resist', 8135: 'Treasure Hunter', 8120: 'Ghost Poro', 8134: 'Ingenious Hunter', 8351: 'Glacial Augment', 8242: 'Unflinching', 8401: 'Shield Bash', 9111: 'Triumph', 8105: 'Relentless Hunter', 8454: 'The Leviathan', 8275: 'Nimbus Cloak', 82